# Neural Identifier Training with Particle Filters - Double Pendulum

In [43]:
import numpy as np
import plotly.graph_objects as go

In [44]:
# ============================================================
# 1) True nonlinear system (double pendulum)
# ============================================================
def plant_dynamics(x, u, m1=1.0, m2=1.0, l1=1.0, l2=1.0, g=9.81, b1=0.1, b2=0.1):
    """
    Continuous dynamics for double pendulum: x = [theta1, omega1, theta2, omega2]. 
    Returns x_dot.
    """
    theta1, omega1, theta2, omega2 = x
    
    # Auxiliary variables
    delta_theta = theta2 - theta1
    den1 = (m1 + m2) * l1 - m2 * l1 * np.cos(delta_theta) * np.cos(delta_theta)
    den2 = (l2 / l1) * den1
    
    # First pendulum equations
    num1 = (-m2 * l1 * omega1**2 * np.sin(delta_theta) * np.cos(delta_theta) + 
            m2 * g * np.sin(theta2) * np.cos(delta_theta) + 
            m2 * l2 * omega2**2 * np.sin(delta_theta) - 
            (m1 + m2) * g * np.sin(theta1) - 
            b1 * omega1)
    
    # Second pendulum equations  
    num2 = (-m2 * l2 * omega2**2 * np.sin(delta_theta) * np.cos(delta_theta) + 
            (m1 + m2) * g * np.sin(theta1) * np.cos(delta_theta) + 
            (m1 + m2) * l1 * omega1**2 * np.sin(delta_theta) - 
            (m1 + m2) * g * np.sin(theta2) - 
            b2 * omega2)
    
    # State derivatives
    theta1_dot = omega1
    omega1_dot = num1 / den1
    theta2_dot = omega2
    omega2_dot = num2 / den2
    
    return np.array([theta1_dot, omega1_dot, theta2_dot, omega2_dot])

def plant(x_k, u_k, dt=0.01, process_noise_type='laplacian', process_noise_std=1e-3):
    """
    One Euler step of the discrete plant with process noise.
    """
    x_dot = plant_dynamics(x_k, u_k)
    x_kp1 = x_k + dt * x_dot

    # Add process noise
    if process_noise_type == 'laplacian':
        noise = np.random.laplace(0, process_noise_std, size=x_kp1.shape)
    elif process_noise_type == 'uniform':
        a = np.sqrt(3) * process_noise_std
        noise = np.random.uniform(-a, a, size=x_kp1.shape)
    else:  # gaussian
        noise = np.random.normal(0, process_noise_std, size=x_kp1.shape)

    return x_kp1 + noise

In [45]:
# ============================================================
# 2) RHONN structure
# ============================================================
def sigmoidal(z, beta=1.0):
    """Sigmoid S(z)."""
    z = np.clip(z, -500, 500)
    return 1.0 / (1.0 + np.exp(-beta * z))

def construct_z_vector(x_est):
    """
    Features for a 4-state double pendulum system (no inputs):
    z = [S(x1), S(x2), S(x3), S(x4), S(x1)S(x2), S(x1)S(x3), S(x1)S(x4), 
         S(x2)S(x3), S(x2)S(x4), S(x3)S(x4), S(x1)^2, S(x2)^2, S(x3)^2, S(x4)^2, 1]
    """
    s_x1 = sigmoidal(x_est[0])  # theta1
    s_x2 = sigmoidal(x_est[1])  # omega1
    s_x3 = sigmoidal(x_est[2])  # theta2
    s_x4 = sigmoidal(x_est[3])  # omega2
    
    return np.array([
        s_x1, s_x2, s_x3, s_x4,                    # Linear terms
        s_x1*s_x2, s_x1*s_x3, s_x1*s_x4,          # Cross terms x1
        s_x2*s_x3, s_x2*s_x4, s_x3*s_x4,          # Cross terms others
        s_x1**2, s_x2**2, s_x3**2, s_x4**2,       # Quadratic terms
        1.0                                         # Bias
    ])


def RHONN_predict(x_state_for_z, w_neuron):
    """
    Predicts next-state component with a single RHONN neuron:
    x_i(k+1) = w_i^T z( x(k) , u(k) )
    """
    z_i = construct_z_vector(x_state_for_z)
    if len(z_i) != len(w_neuron):
        raise ValueError(f"Dimension mismatch: z({len(z_i)}) vs w({len(w_neuron)})")
    return np.dot(w_neuron, z_i)

In [46]:
# ============================================================
# 3) EKF trainer over weights
# ============================================================
class EKF_RHONN_Trainer:
    """
    EKF on each neuron's weight vector, with random-walk weight dynamics.
    Critical: update uses chi_{k+1} as measurement; z is built from time k (series-parallel).
    """
    def __init__(self, num_neurons, num_weights_per_neuron, initial_weights=None,
                 Q_init=1e-4, R_init=1e-2, P_init=1.0, eta=1.0):
        self.num_neurons = num_neurons
        self.num_weights_per_neuron = num_weights_per_neuron
        self.eta = eta

        self.weights = []
        self.P = []
        self.Q = []
        self.R = []

        for i in range(num_neurons):
            if initial_weights is not None and i < len(initial_weights):
                w_i = np.copy(initial_weights[i])
            else:
                w_i = np.random.randn(num_weights_per_neuron) * 0.1
            self.weights.append(w_i)

            self.P.append(np.eye(num_weights_per_neuron) * P_init)
            self.Q.append(np.eye(num_weights_per_neuron) * Q_init)
            self.R.append(np.array([R_init]))

    def update(self, chi_kp1, chi_k, x_hat_previous):
        """
        One EKF update for all neurons.

        chi_kp1: np.array, measured true states at time k+1  (target)
        chi_k  : np.array, measured true states at time k    (for building z)
        x_hat_previous: np.array, previous estimate (time k) to complete z (series-parallel)
        """
        # Build series-parallel state for z: replace measured outputs at time k
        x_state_for_z = np.copy(x_hat_previous)
        x_state_for_z[0] = chi_k[0]  # theta1
        x_state_for_z[2] = chi_k[2]  # theta2 (measured outputs for double pendulum)

        z_i = construct_z_vector(x_state_for_z)          # shape (num_features,)
        H_i = z_i.reshape(-1, 1)                          # column vector

        for i in range(self.num_neurons):
            # Predict covariance
            P_pred = self.P[i] + self.Q[i]

            # Innovation covariance (scalar)
            M_i = self.R[i][0] + (H_i.T @ P_pred @ H_i)[0, 0]
            if M_i < 1e-12:
                M_i = 1e-12

            # Predicted output for neuron i
            x_hat_pred_i = self.weights[i] @ z_i

            # Innovation: measured chi at k+1 minus prediction built from z at k
            e_i = chi_kp1[i] - x_hat_pred_i

            # Kalman gain (flatten to 1D)
            K_i = (P_pred @ H_i).flatten() / M_i

            # Weight update
            self.weights[i] += self.eta * K_i * e_i

            # Covariance update (Joseph or simple; we symmetrize to keep numerical hygiene)
            P_update = P_pred - np.outer(K_i, (H_i.T @ P_pred).ravel())
            self.P[i] = 0.5 * (P_update + P_update.T)  # enforce symmetry

In [47]:
# ============================================================
# 4) Particle Filter trainer over weights
# ============================================================
class PF_RHONN_Trainer:
    """
    Particle filter over neuron weights (per neuron).
    - Predict (random walk on weights)
    - Update (likelihood from chi_{k+1} vs prediction built with z at k)
    - ESS-triggered resampling
    """
    def __init__(self, num_neurons, num_weights_per_neuron, n_particles=100,
                 initial_weights=None, Q_std=0.05, R_std=0.1, ess_threshold=None):
        self.num_neurons = num_neurons
        self.num_weights_per_neuron = num_weights_per_neuron
        self.n_particles = n_particles
        self.Q_std = Q_std
        self.R_std = R_std
        self.R_var = R_std**2
        self.ess_threshold = ess_threshold if ess_threshold is not None else n_particles / 2.0

        self.particles = []
        self.weights_pf = []

        for i in range(num_neurons):
            if initial_weights is not None and i < len(initial_weights):
                base = np.copy(initial_weights[i])
                particles_i = base + np.random.randn(n_particles, num_weights_per_neuron) * 0.1
            else:
                particles_i = np.random.randn(n_particles, num_weights_per_neuron) * 0.1
            self.particles.append(particles_i)
            self.weights_pf.append(np.ones(n_particles) / n_particles)

    def _ess(self, w):
        w = w / np.sum(w)
        return 1.0 / np.sum(w**2)

    def _resample_systematic(self, neuron_index):
        w = self.weights_pf[neuron_index]
        p = self.particles[neuron_index]

        w = w / np.sum(w)
        N = len(w)
        u0 = np.random.uniform(0.0, 1.0 / N)
        cdf = np.cumsum(w)

        indexes = np.zeros(N, dtype=int)
        i, j = 0, 0
        while i < N:
            u = u0 + i / N
            while u > cdf[j]:
                j += 1
            indexes[i] = j
            i += 1

        self.particles[neuron_index] = p[indexes]
        self.weights_pf[neuron_index] = np.ones(N) / N

    def update(self, chi_kp1, chi_k, x_hat_previous):
        """
        One PF step over all neuron weight-sets.

        chi_kp1: measured true states at k+1 (targets)
        chi_k  : measured true states at k   (for z)
        x_hat_previous: previous estimate at k (to complete z)
        """
        # Build z from time k (series-parallel)
        x_state_for_z = np.copy(x_hat_previous)
        x_state_for_z[0] = chi_k[0]  # theta1
        x_state_for_z[2] = chi_k[2]  # theta2 (measured outputs for double pendulum)
        z = construct_z_vector(x_state_for_z)  # (num_features,)

        # 1) Predict: random walk on weights
        for i in range(self.num_neurons):
            self.particles[i] += np.random.randn(self.n_particles, self.num_weights_per_neuron) * self.Q_std

        # 2) Update: importance weights with Gaussian likelihood
        for i in range(self.num_neurons):
            w_mat = self.particles[i]                               # (N, num_features)
            x_pred_particles = w_mat @ z                            # (N,)
            innov = chi_kp1[i] - x_pred_particles

            # Log-likelihood for stability
            ll = -0.5 * (innov**2) / self.R_var
            ll -= np.max(ll)
            like = np.exp(ll)

            self.weights_pf[i] *= like
            s = np.sum(self.weights_pf[i])
            if s < 1e-300:
                # Weight collapse safeguard
                self.weights_pf[i] = np.ones(self.n_particles) / self.n_particles
            else:
                self.weights_pf[i] /= s

            # 3) Resample if ESS is low
            if self._ess(self.weights_pf[i]) < self.ess_threshold:
                self._resample_systematic(i)

    def get_estimate(self):
        """Mean of particles per neuron (after any resampling)."""
        return [np.mean(self.particles[i], axis=0) for i in range(self.num_neurons)]


In [48]:
# ============================================================
# 5) Simulation
# ============================================================
if __name__ == "__main__":
    # --- Simulation settings ---
    n_steps = 1000
    dt = 0.01
    t_history = np.linspace(0, (n_steps-1) * dt, n_steps)

    process_noise_type = 'laplacian'  # 'laplacian' | 'uniform' | 'gaussian'
    process_noise_std = 0.01

    # --- True system init ---
    x_true = np.zeros((n_steps, 4))
    x_true[0] = [np.pi / 6, 0.0, np.pi / 4, 0.0]  # theta1=30°, omega1=0, theta2=45°, omega2=0
    u = 0.0

    # --- RHONN config ---
    num_neurons = 4  # Four states for double pendulum
    num_features = 15  # Updated feature vector size
    num_weights_per_neuron = num_features

    # --- Common initial weights for fair comparison ---
    # np.random.seed(12345)  # (optional) reproducibility of initial weights
    common_initial_weights = [np.random.uniform(-0.5, 0.5, num_weights_per_neuron) for _ in range(num_neurons)]
    print("Common Initial Weights:")
    for i, w in enumerate(common_initial_weights):
        print(f"  Neuron {i}: {w}")

    # --- EKF ---
    ekf_trainer = EKF_RHONN_Trainer(
        num_neurons, num_weights_per_neuron,
        initial_weights=common_initial_weights,
        Q_init=1e-4, R_init=1e-2, P_init=1.0, eta=1.0
    )
    x_hat_ekf = np.zeros((n_steps, 4))
    x_hat_ekf[0] = x_true[0]

    # --- PF ---
    n_particles = 650
    pf_trainer = PF_RHONN_Trainer(
        num_neurons, num_weights_per_neuron,
        n_particles=n_particles,
        initial_weights=common_initial_weights,
        Q_std=0.5, R_std=np.sqrt(0.001), ess_threshold=n_particles / 2  # ESS < N/2
    )

    # Force identical particle initialization if desired:
    def initialize_pf_with_common_weights(pf_trainer_instance, common_weights_list):
        for i in range(pf_trainer_instance.num_neurons):
            pf_trainer_instance.particles[i] = np.tile(
                common_weights_list[i], (pf_trainer_instance.n_particles, 1)
            )
            pf_trainer_instance.weights_pf[i] = np.ones(pf_trainer_instance.n_particles) / pf_trainer_instance.n_particles

    initialize_pf_with_common_weights(pf_trainer, common_initial_weights)

    x_hat_pf = np.zeros((n_steps, 4))
    x_hat_pf[0] = x_true[0]

    print("Starting simulation...")
    for k in range(n_steps - 1):
        # ---- 1) true system -> k+1 ----
        x_true[k+1] = plant(x_true[k], u, dt, process_noise_type, process_noise_std)

        # ---- 2) EKF update (uses chi_{k+1} target, z from k), then predict x_hat_{k+1} ----
        ekf_trainer.update(chi_kp1=x_true[k+1], chi_k=x_true[k], x_hat_previous=x_hat_ekf[k])

        x_state_for_z_ekf = np.copy(x_hat_ekf[k])
        x_state_for_z_ekf[0] = x_true[k][0]  # series-parallel uses measured theta1 at k
        x_state_for_z_ekf[2] = x_true[k][2]  # series-parallel uses measured theta2 at k
        x_hat_ekf[k+1, 0] = RHONN_predict(x_state_for_z_ekf, ekf_trainer.weights[0])  # theta1
        x_hat_ekf[k+1, 1] = RHONN_predict(x_state_for_z_ekf, ekf_trainer.weights[1])  # omega1
        x_hat_ekf[k+1, 2] = RHONN_predict(x_state_for_z_ekf, ekf_trainer.weights[2])  # theta2
        x_hat_ekf[k+1, 3] = RHONN_predict(x_state_for_z_ekf, ekf_trainer.weights[3])  # omega2

        # ---- 3) PF update (chi_{k+1} vs z from k), then predict x_hat_{k+1} with mean weights ----
        pf_trainer.update(chi_kp1=x_true[k+1], chi_k=x_true[k], x_hat_previous=x_hat_pf[k])

        pf_weight_estimates = pf_trainer.get_estimate()
        x_state_for_z_pf = np.copy(x_hat_pf[k])
        x_state_for_z_pf[0] = x_true[k][0]  # series-parallel uses measured theta1 at k
        x_state_for_z_pf[2] = x_true[k][2]  # series-parallel uses measured theta2 at k
        x_hat_pf[k+1, 0] = RHONN_predict(x_state_for_z_pf, pf_weight_estimates[0])   # theta1
        x_hat_pf[k+1, 1] = RHONN_predict(x_state_for_z_pf, pf_weight_estimates[1])   # omega1
        x_hat_pf[k+1, 2] = RHONN_predict(x_state_for_z_pf, pf_weight_estimates[2])   # theta2
        x_hat_pf[k+1, 3] = RHONN_predict(x_state_for_z_pf, pf_weight_estimates[3])   # omega2

        if k % (n_steps // 10) == 0:
            print(f"Simulation progress: {k/n_steps*100:.1f}%")

    print("Simulation finished.")

Common Initial Weights:
  Neuron 0: [-0.18237906 -0.3438512  -0.38084876  0.21459767  0.26772087 -0.18813535
  0.45764967  0.20779127  0.27408736  0.26000139 -0.46551601 -0.10997911
 -0.37679922  0.17938557 -0.25077444]
  Neuron 1: [ 0.42706728  0.00791653  0.41020314  0.03342245  0.23501346  0.09685571
  0.14271598 -0.3200123  -0.33180857 -0.35623123 -0.09488205 -0.22523376
 -0.06217223  0.27957606  0.45097778]
  Neuron 2: [ 0.21982739  0.30896124  0.22880535  0.43926739  0.109718   -0.41957671
  0.2399628  -0.20802108 -0.32733198 -0.33032654 -0.17105571 -0.26322252
  0.39005564  0.40163325 -0.27894605]
  Neuron 3: [ 0.11051397 -0.14810291 -0.05539696 -0.3930532  -0.47304567 -0.36667398
  0.34964095  0.46512793 -0.42248601  0.1661931  -0.16326359 -0.40054877
  0.46498875 -0.35988219 -0.07423961]
Starting simulation...
Simulation progress: 0.0%
Simulation progress: 10.0%
Simulation progress: 20.0%
Simulation progress: 30.0%
Simulation progress: 20.0%
Simulation progress: 30.0%
Simulati

In [ ]:
 # ============================================================
    # 6) Results & plots for Double Pendulum
# ============================================================
mse_x1_ekf = np.mean((x_true[:, 0] - x_hat_ekf[:, 0])**2)  # theta1
mse_x2_ekf = np.mean((x_true[:, 1] - x_hat_ekf[:, 1])**2)  # omega1
mse_x3_ekf = np.mean((x_true[:, 2] - x_hat_ekf[:, 2])**2)  # theta2
mse_x4_ekf = np.mean((x_true[:, 3] - x_hat_ekf[:, 3])**2)  # omega2

mse_x1_pf = np.mean((x_true[:, 0] - x_hat_pf[:, 0])**2)   # theta1
mse_x2_pf = np.mean((x_true[:, 1] - x_hat_pf[:, 1])**2)   # omega1
mse_x3_pf = np.mean((x_true[:, 2] - x_hat_pf[:, 2])**2)   # theta2
mse_x4_pf = np.mean((x_true[:, 3] - x_hat_pf[:, 3])**2)   # omega2

print(f"\nFinal EKF-RHONN Weights:")
for i in range(4):
    print(f"  Neuron {i+1} (x{i+1}): {ekf_trainer.weights[i]}")

print(f"\nFinal PF-RHONN Weight Estimates:")
pf_estimates = pf_trainer.get_estimate()
for i in range(4):
    print(f"  Neuron {i+1} (x{i+1}): {pf_estimates[i]}")

print("\n--- Performance Comparison (MSE) for Double Pendulum ---")
print(f"EKF MSE x1 (θ₁):        {mse_x1_ekf:.6f}")
print(f"EKF MSE x2 (ω₁):        {mse_x2_ekf:.6f}")
print(f"EKF MSE x3 (θ₂):        {mse_x3_ekf:.6f}")
print(f"EKF MSE x4 (ω₂):        {mse_x4_ekf:.6f}")
print(f"PF  MSE x1 (θ₁):        {mse_x1_pf:.6f}")
print(f"PF  MSE x2 (ω₁):        {mse_x2_pf:.6f}")
print(f"PF  MSE x3 (θ₂):        {mse_x3_pf:.6f}")
print(f"PF  MSE x4 (ω₂):        {mse_x4_pf:.6f}")

states_info = [
    {'idx': 0, 'var': 'θ₁', 'desc': 'First Pendulum Angle', 'y_label': 'Angle (rad)',
     'chi': 'χ₁ (True θ₁)', 'x': 'x₁ (Est. θ₁)'},
    {'idx': 1, 'var': 'ω₁', 'desc': 'First Pendulum Angular Velocity', 'y_label': 'Ang. Vel. (rad/s)',
     'chi': 'χ₂ (True ω₁)', 'x': 'x₂ (Est. ω₁)'},
    {'idx': 2, 'var': 'θ₂', 'desc': 'Second Pendulum Angle', 'y_label': 'Angle (rad)',
     'chi': 'χ₃ (True θ₂)', 'x': 'x₃ (Est. θ₂)'},
    {'idx': 3, 'var': 'ω₂', 'desc': 'Second Pendulum Angular Velocity', 'y_label': 'Ang. Vel. (rad/s)',
     'chi': 'χ₄ (True ω₂)', 'x': 'x₄ (Est. ω₂)'}
]

for state_info in states_info:
    i = state_info['idx']
    trace_plant = go.Scatter(x=t_history, y=x_true[:, i], mode='lines',
                            name=state_info['chi'], line=dict(color='black', width=2))
    trace_pf = go.Scatter(x=t_history, y=x_hat_pf[:, i], mode='lines',
                        name=f"{state_info['x']} (PF)", line=dict(dash='dot'))
    trace_ekf = go.Scatter(x=t_history, y=x_hat_ekf[:, i], mode='lines',
                        name=f"{state_info['x']} (EKF)", line=dict(dash='dash'))

    fig = go.Figure([trace_plant, trace_pf, trace_ekf])
    fig.update_layout(
        title=f'Double Pendulum RHONN Identification for {state_info["var"]} ({state_info["desc"]})',
        xaxis_title='Time (s)',
        yaxis_title=state_info['y_label'],
        legend=dict(x=0, y=1, orientation='h'),
        font=dict(size=12),
        plot_bgcolor='white',
        paper_bgcolor='white'
    )
    fig.show()

# Errors for all four states
error_x1_ekf = x_true[:, 0] - x_hat_ekf[:, 0]
error_x2_ekf = x_true[:, 1] - x_hat_ekf[:, 1]
error_x3_ekf = x_true[:, 2] - x_hat_ekf[:, 2]
error_x4_ekf = x_true[:, 3] - x_hat_ekf[:, 3]

error_x1_pf = x_true[:, 0] - x_hat_pf[:, 0]
error_x2_pf = x_true[:, 1] - x_hat_pf[:, 1]
error_x3_pf = x_true[:, 2] - x_hat_pf[:, 2]
error_x4_pf = x_true[:, 3] - x_hat_pf[:, 3]

fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=t_history, y=error_x1_ekf, mode='lines',
                        name=f'EKF Error θ₁ (MSE={mse_x1_ekf:.6f})', opacity=0.7))
fig2.add_trace(go.Scatter(x=t_history, y=error_x1_pf, mode='lines',
                        name=f'PF Error θ₁ (MSE={mse_x1_pf:.6f})', opacity=0.7))
fig2.add_trace(go.Scatter(x=t_history, y=error_x2_ekf, mode='lines',
                        name=f'EKF Error ω₁ (MSE={mse_x2_ekf:.6f})', opacity=0.7))
fig2.add_trace(go.Scatter(x=t_history, y=error_x2_pf, mode='lines',
                        name=f'PF Error ω₁ (MSE={mse_x2_pf:.6f})', opacity=0.7))
fig2.update_layout(
    title='Double Pendulum Identification Errors - First Pendulum',
    xaxis_title='Time (s)',
    yaxis_title='Error',
    legend=dict(x=0, y=1, orientation='h'),
    font=dict(size=12),
    plot_bgcolor='white',
    paper_bgcolor='white'
)
fig2.show()

# Second pendulum errors
fig3 = go.Figure()
fig3.add_trace(go.Scatter(x=t_history, y=error_x3_ekf, mode='lines',
                        name=f'EKF Error θ₂ (MSE={mse_x3_ekf:.6f})', opacity=0.7))
fig3.add_trace(go.Scatter(x=t_history, y=error_x3_pf, mode='lines',
                        name=f'PF Error θ₂ (MSE={mse_x3_pf:.6f})', opacity=0.7))
fig3.add_trace(go.Scatter(x=t_history, y=error_x4_ekf, mode='lines',
                        name=f'EKF Error ω₂ (MSE={mse_x4_ekf:.6f})', opacity=0.7))
fig3.add_trace(go.Scatter(x=t_history, y=error_x4_pf, mode='lines',
                        name=f'PF Error ω₂ (MSE={mse_x4_pf:.6f})', opacity=0.7))
fig3.update_layout(
    title='Double Pendulum Identification Errors - Second Pendulum',
    xaxis_title='Time (s)',
    yaxis_title='Error',
    legend=dict(x=0, y=1, orientation='h'),
    font=dict(size=12),
    plot_bgcolor='white',
    paper_bgcolor='white'
)
fig3.show()


Final EKF-RHONN Weights:
  Neuron 1 (x1): [ 1.85786447 -0.40558681 -0.31127962 -0.47183847  0.3612191   0.77675647
  0.55672854 -0.16506691  0.13335386  0.01771277  1.33638258  0.26722079
  0.01908849  0.00437771 -1.25061809]
  Neuron 2 (x2): [-1.44071904e+00  4.02219120e+00  2.75746860e-01 -8.05948695e-01
  1.51979176e+00  4.65283048e-01 -1.16852059e+00 -1.23501230e+00
  4.50480033e-01  1.20493292e+00 -1.59445523e-03  3.16421002e-01
  1.02554922e-01  2.16563852e-01 -1.45084530e+00]
  Neuron 3 (x3): [-0.10928276 -0.15301411  2.12588954 -0.31191997 -0.05290172  0.3981429
 -0.05455367  0.22534435  0.17930524  0.13573218 -0.0741112   0.00671274
  1.58654371  0.25200144 -1.4627497 ]
  Neuron 4 (x4): [-0.65292368  2.06801501 -0.44408946  3.43803016 -3.64644832  0.23261483
  3.58554074 -2.71127206  0.06666344 -0.1447426   0.67191337 -0.03452821
  1.38639829 -0.59348403 -1.87502036]

Final PF-RHONN Weight Estimates:
  Neuron 1 (x1): [ -9.78569858  25.58932923   6.00026719   5.73091312  -8.32